# Foresight tier medallion - SIT evidence run

Sprint 26 WS-A (issue #335). Deterministically materializes the three
WS-A gold tables - `gold.fact_occupancy_forecast`, `gold.fact_forecast_driver`,
`gold.fact_signal` - so the Foresight tier can be proven live in the SIT
lakehouse. Synthetic-only, deterministic, no PHI (ADR-0013 / ADR-0016).

Generated by `data-platform/scripts/fabric/build_forecast_evidence_notebook.py` -
do not edit by hand; re-generate to update.


In [ ]:
from pyspark.sql.types import (
    ArrayType, BooleanType, DoubleType, LongType, StringType,
    StructField, StructType,
)

FORECAST_SCHEMA = StructType([
    StructField("contractId", StringType(), True),
    StructField("forecastId", StringType(), True),
    StructField("hospitalId", StringType(), True),
    StructField("wardId", StringType(), True),
    StructField("producedAt", StringType(), True),
    StructField("producedBy", StringType(), True),
    StructField("modelVersion", StringType(), True),
    StructField("horizonH", LongType(), True),
    StructField("bucketStart", StringType(), True),
    StructField("bedCapacity", DoubleType(), True),
    StructField("forecastOccupiedBeds", DoubleType(), True),
    StructField("forecastOccupancyPct", DoubleType(), True),
    StructField("lowerCi", DoubleType(), True),
    StructField("upperCi", DoubleType(), True),
    StructField("breach", BooleanType(), True),
    StructField("purposeTag", StringType(), True),
    StructField("dataResidencyRegion", StringType(), True),
    StructField("asOfTimestamp", StringType(), True),
])

DRIVER_SCHEMA = StructType([
    StructField("contractId", StringType(), True),
    StructField("forecastId", StringType(), True),
    StructField("hospitalId", StringType(), True),
    StructField("wardId", StringType(), True),
    StructField("horizonH", LongType(), True),
    StructField("factor", StringType(), True),
    StructField("delta", DoubleType(), True),
    StructField("note", StringType(), True),
    StructField("signalId", StringType(), True),
    StructField("purposeTag", StringType(), True),
    StructField("asOfTimestamp", StringType(), True),
])

SIGNAL_SCHEMA = StructType([
    StructField("signal_id", StringType(), True),
    StructField("source_id", StringType(), True),
    StructField("hazard_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("trust_tier", StringType(), True),
    StructField("probability", DoubleType(), True),
    StructField("evidences_factor", StringType(), True),
    StructField("cantons", ArrayType(StringType()), True),
    StructField("onset", StringType(), True),
])


def _write(rows, schema, table):
    df = spark.createDataFrame(rows, schema)
    df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table)
    print(f"wrote {table}: {df.count()} rows")


In [ ]:
# Gold - deterministic 72h occupancy forecast (one row per ward x horizon)
forecast_rows = [['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 0, '2026-07-23T00:00:00Z', 50.0, 51.0, 102.0, 48.45, 53.55, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 1, '2026-07-23T01:00:00Z', 50.0, 51.055, 102.11, 48.396, 53.714, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 2, '2026-07-23T02:00:00Z', 50.0, 51.111, 102.222, 48.342, 53.88, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 3, '2026-07-23T03:00:00Z', 50.0, 51.167, 102.334, 48.289, 54.045, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 4, '2026-07-23T04:00:00Z', 50.0, 51.222, 102.444, 48.234, 54.21, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 5, '2026-07-23T05:00:00Z', 50.0, 51.278, 102.556, 48.18, 54.376, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 6, '2026-07-23T06:00:00Z', 50.0, 51.333, 102.666, 48.125, 54.541, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 7, '2026-07-23T07:00:00Z', 50.0, 51.389, 102.778, 48.07, 54.708, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 8, '2026-07-23T08:00:00Z', 50.0, 51.445, 102.89, 48.015, 54.875, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 9, '2026-07-23T09:00:00Z', 50.0, 51.5, 103.0, 47.959, 55.041, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 10, '2026-07-23T10:00:00Z', 50.0, 51.555, 103.11, 47.903, 55.207, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 11, '2026-07-23T11:00:00Z', 50.0, 51.611, 103.222, 47.848, 55.374, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 12, '2026-07-23T12:00:00Z', 50.0, 51.667, 103.334, 47.792, 55.542, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 13, '2026-07-23T13:00:00Z', 50.0, 51.722, 103.444, 47.735, 55.709, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 14, '2026-07-23T14:00:00Z', 50.0, 51.778, 103.556, 47.679, 55.877, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 15, '2026-07-23T15:00:00Z', 50.0, 51.833, 103.666, 47.622, 56.044, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 16, '2026-07-23T16:00:00Z', 50.0, 51.889, 103.778, 47.565, 56.213, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 17, '2026-07-23T17:00:00Z', 50.0, 51.945, 103.89, 47.508, 56.382, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 18, '2026-07-23T18:00:00Z', 50.0, 52.0, 104.0, 47.45, 56.55, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 19, '2026-07-23T19:00:00Z', 50.0, 52.055, 104.11, 47.392, 56.718, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 20, '2026-07-23T20:00:00Z', 50.0, 52.111, 104.222, 47.334, 56.888, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 21, '2026-07-23T21:00:00Z', 50.0, 52.167, 104.334, 47.276, 57.058, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 22, '2026-07-23T22:00:00Z', 50.0, 52.222, 104.444, 47.217, 57.227, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 23, '2026-07-23T23:00:00Z', 50.0, 52.278, 104.556, 47.159, 57.397, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 24, '2026-07-24T00:00:00Z', 50.0, 52.333, 104.666, 47.1, 57.566, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 25, '2026-07-24T01:00:00Z', 50.0, 52.389, 104.778, 47.041, 57.737, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 26, '2026-07-24T02:00:00Z', 50.0, 52.445, 104.89, 46.982, 57.908, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 27, '2026-07-24T03:00:00Z', 50.0, 52.5, 105.0, 46.922, 58.078, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 28, '2026-07-24T04:00:00Z', 50.0, 52.555, 105.11, 46.862, 58.248, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 29, '2026-07-24T05:00:00Z', 50.0, 52.611, 105.222, 46.802, 58.42, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 30, '2026-07-24T06:00:00Z', 50.0, 52.667, 105.334, 46.742, 58.592, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 31, '2026-07-24T07:00:00Z', 50.0, 52.722, 105.444, 46.681, 58.763, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 32, '2026-07-24T08:00:00Z', 50.0, 52.778, 105.556, 46.621, 58.935, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 33, '2026-07-24T09:00:00Z', 50.0, 52.833, 105.666, 46.559, 59.107, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 34, '2026-07-24T10:00:00Z', 50.0, 52.889, 105.778, 46.498, 59.28, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 35, '2026-07-24T11:00:00Z', 50.0, 52.945, 105.89, 46.437, 59.453, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 36, '2026-07-24T12:00:00Z', 50.0, 53.0, 106.0, 46.375, 59.625, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 37, '2026-07-24T13:00:00Z', 50.0, 53.055, 106.11, 46.313, 59.797, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 38, '2026-07-24T14:00:00Z', 50.0, 53.111, 106.222, 46.251, 59.971, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 39, '2026-07-24T15:00:00Z', 50.0, 53.167, 106.334, 46.189, 60.145, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 40, '2026-07-24T16:00:00Z', 50.0, 53.222, 106.444, 46.126, 60.318, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 41, '2026-07-24T17:00:00Z', 50.0, 53.278, 106.556, 46.063, 60.493, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 42, '2026-07-24T18:00:00Z', 50.0, 53.333, 106.666, 46.0, 60.666, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 43, '2026-07-24T19:00:00Z', 50.0, 53.389, 106.778, 45.937, 60.841, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 44, '2026-07-24T20:00:00Z', 50.0, 53.445, 106.89, 45.874, 61.016, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 45, '2026-07-24T21:00:00Z', 50.0, 53.5, 107.0, 45.809, 61.191, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 46, '2026-07-24T22:00:00Z', 50.0, 53.555, 107.11, 45.745, 61.365, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 47, '2026-07-24T23:00:00Z', 50.0, 53.611, 107.222, 45.681, 61.541, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 48, '2026-07-25T00:00:00Z', 50.0, 53.667, 107.334, 45.617, 61.717, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 49, '2026-07-25T01:00:00Z', 50.0, 53.722, 107.444, 45.552, 61.892, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 50, '2026-07-25T02:00:00Z', 50.0, 53.778, 107.556, 45.487, 62.069, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 51, '2026-07-25T03:00:00Z', 50.0, 53.833, 107.666, 45.422, 62.244, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 52, '2026-07-25T04:00:00Z', 50.0, 53.889, 107.778, 45.357, 62.421, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 53, '2026-07-25T05:00:00Z', 50.0, 53.945, 107.89, 45.291, 62.599, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 54, '2026-07-25T06:00:00Z', 50.0, 54.0, 108.0, 45.225, 62.775, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 55, '2026-07-25T07:00:00Z', 50.0, 54.055, 108.11, 45.158, 62.952, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 56, '2026-07-25T08:00:00Z', 50.0, 54.111, 108.222, 45.092, 63.13, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 57, '2026-07-25T09:00:00Z', 50.0, 54.167, 108.334, 45.026, 63.308, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 58, '2026-07-25T10:00:00Z', 50.0, 54.222, 108.444, 44.959, 63.485, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 59, '2026-07-25T11:00:00Z', 50.0, 54.278, 108.556, 44.892, 63.664, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 60, '2026-07-25T12:00:00Z', 50.0, 54.333, 108.666, 44.825, 63.841, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 61, '2026-07-25T13:00:00Z', 50.0, 54.389, 108.778, 44.758, 64.02, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 62, '2026-07-25T14:00:00Z', 50.0, 54.445, 108.89, 44.69, 64.2, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 63, '2026-07-25T15:00:00Z', 50.0, 54.5, 109.0, 44.622, 64.378, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 64, '2026-07-25T16:00:00Z', 50.0, 54.555, 109.11, 44.553, 64.557, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 65, '2026-07-25T17:00:00Z', 50.0, 54.611, 109.222, 44.485, 64.737, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 66, '2026-07-25T18:00:00Z', 50.0, 54.667, 109.334, 44.417, 64.917, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 67, '2026-07-25T19:00:00Z', 50.0, 54.722, 109.444, 44.348, 65.096, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 68, '2026-07-25T20:00:00Z', 50.0, 54.778, 109.556, 44.279, 65.277, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 69, '2026-07-25T21:00:00Z', 50.0, 54.833, 109.666, 44.209, 65.457, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 70, '2026-07-25T22:00:00Z', 50.0, 54.889, 109.778, 44.14, 65.638, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 71, '2026-07-25T23:00:00Z', 50.0, 54.945, 109.89, 44.07, 65.82, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z'], ['DC-OCCUPANCY-FORECAST-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', '2026-07-23T00:00:00Z', 'MRUN-FORESIGHT-SYNTH-V0-1', '0.1.0', 72, '2026-07-26T00:00:00Z', 50.0, 55.0, 110.0, 44.0, 66.0, True, 'capacity-planning', 'switzerlandnorth', '2026-07-23T00:00:00Z']]
_write(forecast_rows, FORECAST_SCHEMA, "gold.fact_occupancy_forecast")


In [ ]:
# Gold - forecast driver decomposition (the 'why'; deltas reconcile to net change)
driver_rows = [['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 0, 'forecast_admissions', 0.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 0, 'planned_discharges', -0.0, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 0, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 0, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 1, 'forecast_admissions', 0.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 1, 'planned_discharges', -0.028, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 1, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 1, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 2, 'forecast_admissions', 0.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 2, 'planned_discharges', -0.056, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 2, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 2, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 3, 'forecast_admissions', 0.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 3, 'planned_discharges', -0.083, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 3, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 3, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 4, 'forecast_admissions', 0.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 4, 'planned_discharges', -0.111, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 4, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 4, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 5, 'forecast_admissions', 0.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 5, 'planned_discharges', -0.139, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 5, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 5, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 6, 'forecast_admissions', 0.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 6, 'planned_discharges', -0.167, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 6, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 6, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 7, 'forecast_admissions', 0.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 7, 'planned_discharges', -0.194, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 7, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 7, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 8, 'forecast_admissions', 0.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 8, 'planned_discharges', -0.222, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 8, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 8, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 9, 'forecast_admissions', 0.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 9, 'planned_discharges', -0.25, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 9, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 9, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 10, 'forecast_admissions', 0.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 10, 'planned_discharges', -0.278, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 10, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 10, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 11, 'forecast_admissions', 0.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 11, 'planned_discharges', -0.306, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 11, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 11, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 12, 'forecast_admissions', 1.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 12, 'planned_discharges', -0.333, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 12, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 12, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 13, 'forecast_admissions', 1.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 13, 'planned_discharges', -0.361, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 13, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 13, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 14, 'forecast_admissions', 1.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 14, 'planned_discharges', -0.389, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 14, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 14, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 15, 'forecast_admissions', 1.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 15, 'planned_discharges', -0.417, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 15, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 15, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 16, 'forecast_admissions', 1.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 16, 'planned_discharges', -0.444, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 16, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 16, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 17, 'forecast_admissions', 1.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 17, 'planned_discharges', -0.472, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 17, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 17, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 18, 'forecast_admissions', 1.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 18, 'planned_discharges', -0.5, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 18, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 18, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 19, 'forecast_admissions', 1.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 19, 'planned_discharges', -0.528, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 19, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 19, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 20, 'forecast_admissions', 1.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 20, 'planned_discharges', -0.556, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 20, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 20, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 21, 'forecast_admissions', 1.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 21, 'planned_discharges', -0.583, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 21, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 21, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 22, 'forecast_admissions', 1.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 22, 'planned_discharges', -0.611, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 22, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 22, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 23, 'forecast_admissions', 1.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 23, 'planned_discharges', -0.639, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 23, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 23, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 24, 'forecast_admissions', 2.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 24, 'planned_discharges', -0.667, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 24, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 24, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 25, 'forecast_admissions', 2.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 25, 'planned_discharges', -0.694, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 25, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 25, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 26, 'forecast_admissions', 2.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 26, 'planned_discharges', -0.722, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 26, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 26, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 27, 'forecast_admissions', 2.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 27, 'planned_discharges', -0.75, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 27, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 27, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 28, 'forecast_admissions', 2.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 28, 'planned_discharges', -0.778, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 28, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 28, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 29, 'forecast_admissions', 2.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 29, 'planned_discharges', -0.806, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 29, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 29, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 30, 'forecast_admissions', 2.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 30, 'planned_discharges', -0.833, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 30, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 30, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 31, 'forecast_admissions', 2.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 31, 'planned_discharges', -0.861, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 31, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 31, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 32, 'forecast_admissions', 2.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 32, 'planned_discharges', -0.889, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 32, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 32, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 33, 'forecast_admissions', 2.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 33, 'planned_discharges', -0.917, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 33, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 33, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 34, 'forecast_admissions', 2.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 34, 'planned_discharges', -0.944, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 34, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 34, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 35, 'forecast_admissions', 2.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 35, 'planned_discharges', -0.972, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 35, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 35, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 36, 'forecast_admissions', 3.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 36, 'planned_discharges', -1.0, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 36, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 36, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 37, 'forecast_admissions', 3.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 37, 'planned_discharges', -1.028, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 37, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 37, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 38, 'forecast_admissions', 3.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 38, 'planned_discharges', -1.056, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 38, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 38, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 39, 'forecast_admissions', 3.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 39, 'planned_discharges', -1.083, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 39, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 39, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 40, 'forecast_admissions', 3.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 40, 'planned_discharges', -1.111, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 40, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 40, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 41, 'forecast_admissions', 3.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 41, 'planned_discharges', -1.139, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 41, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 41, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 42, 'forecast_admissions', 3.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 42, 'planned_discharges', -1.167, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 42, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 42, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 43, 'forecast_admissions', 3.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 43, 'planned_discharges', -1.194, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 43, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 43, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 44, 'forecast_admissions', 3.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 44, 'planned_discharges', -1.222, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 44, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 44, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 45, 'forecast_admissions', 3.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 45, 'planned_discharges', -1.25, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 45, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 45, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 46, 'forecast_admissions', 3.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 46, 'planned_discharges', -1.278, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 46, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 46, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 47, 'forecast_admissions', 3.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 47, 'planned_discharges', -1.306, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 47, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 47, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 48, 'forecast_admissions', 4.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 48, 'planned_discharges', -1.333, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 48, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 48, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 49, 'forecast_admissions', 4.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 49, 'planned_discharges', -1.361, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 49, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 49, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 50, 'forecast_admissions', 4.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 50, 'planned_discharges', -1.389, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 50, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 50, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 51, 'forecast_admissions', 4.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 51, 'planned_discharges', -1.417, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 51, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 51, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 52, 'forecast_admissions', 4.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 52, 'planned_discharges', -1.444, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 52, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 52, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 53, 'forecast_admissions', 4.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 53, 'planned_discharges', -1.472, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 53, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 53, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 54, 'forecast_admissions', 4.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 54, 'planned_discharges', -1.5, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 54, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 54, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 55, 'forecast_admissions', 4.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 55, 'planned_discharges', -1.528, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 55, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 55, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 56, 'forecast_admissions', 4.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 56, 'planned_discharges', -1.556, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 56, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 56, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 57, 'forecast_admissions', 4.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 57, 'planned_discharges', -1.583, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 57, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 57, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 58, 'forecast_admissions', 4.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 58, 'planned_discharges', -1.611, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 58, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 58, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 59, 'forecast_admissions', 4.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 59, 'planned_discharges', -1.639, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 59, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 59, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 60, 'forecast_admissions', 5.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 60, 'planned_discharges', -1.667, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 60, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 60, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 61, 'forecast_admissions', 5.083, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 61, 'planned_discharges', -1.694, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 61, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 61, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 62, 'forecast_admissions', 5.167, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 62, 'planned_discharges', -1.722, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 62, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 62, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 63, 'forecast_admissions', 5.25, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 63, 'planned_discharges', -1.75, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 63, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 63, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 64, 'forecast_admissions', 5.333, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 64, 'planned_discharges', -1.778, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 64, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 64, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 65, 'forecast_admissions', 5.417, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 65, 'planned_discharges', -1.806, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 65, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 65, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 66, 'forecast_admissions', 5.5, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 66, 'planned_discharges', -1.833, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 66, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 66, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 67, 'forecast_admissions', 5.583, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 67, 'planned_discharges', -1.861, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 67, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 67, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 68, 'forecast_admissions', 5.667, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 68, 'planned_discharges', -1.889, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 68, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 68, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 69, 'forecast_admissions', 5.75, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 69, 'planned_discharges', -1.917, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 69, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 69, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 70, 'forecast_admissions', 5.833, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 70, 'planned_discharges', -1.944, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 70, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 70, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 71, 'forecast_admissions', 5.917, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 71, 'planned_discharges', -1.972, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 71, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 71, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 72, 'forecast_admissions', 6.0, 'forecast admissions', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 72, 'planned_discharges', -2.0, 'planned discharges', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 72, 'transfers', 0.0, 'net transfers', None, 'capacity-planning', '2026-07-23T00:00:00Z'], ['DC-FORECAST-DRIVER-v1', 'OF-H-USZ-MEDICINE-A-20260723T00', 'H_USZ', 'Medicine A', 72, 'seasonality', 0.0, 'flu season', 'cap-2026-flu-zh-1', 'capacity-planning', '2026-07-23T00:00:00Z']]
_write(driver_rows, DRIVER_SCHEMA, "gold.fact_forecast_driver")


In [ ]:
# Gold - deny-by-default Trust-A signal projection over the S21 ext spine
signal_rows = [['cap-2026-heat-zh-1', 'alertswiss', 'heat', 'Severe', 'A', 0.9, 'seasonality', ['ZH'], '2026-07-17T12:00:00Z'], ['bag-rsv-2026-w29', 'bag', 'rsv', 'Moderate', 'A', 0.6, 'seasonality', ['BE', 'ZH'], '2026-07-13T00:00:00Z'], ['ms-heat-2026-0001', 'meteoswiss', 'heat', 'Severe', 'A', 0.9, 'seasonality', ['ZH'], '2026-07-17T12:00:00Z'], ['sed-2026-0007', 'sed', 'earthquake', 'Severe', 'A', 0.9, 'seasonality', ['VS'], '2026-07-17T10:00:00Z']]
_write(signal_rows, SIGNAL_SCHEMA, "gold.fact_signal")


In [ ]:
# Inline verification - print counts for the evidence doc
for t in ["gold.fact_occupancy_forecast", "gold.fact_forecast_driver",
          "gold.fact_signal"]:
    print(t, spark.table(t).count())
display(spark.table("gold.fact_occupancy_forecast").where("horizonH = 72").select(
    "wardId", "horizonH", "forecastOccupiedBeds", "forecastOccupancyPct", "breach"))
